## Access dataset details

`get_dataset()` returns a `Dataset` object describing a Cecil dataset — its variables, constraints, and provenance. Use it to inspect what a dataset contains before creating subscriptions, or to read per-variable attributes such as `no_data` values during analysis.

In [2]:
import cecil

client = cecil.Client()

### Fetch a dataset

Pass a dataset ID to `get_dataset()`. Dataset IDs are available in the [Cecil dataset catalogue](https://docs.cecil.earth/datasets) or by calling `client.list_datasets()`.

In [3]:
dataset = client.get_dataset('2ce9cb24-dba0-4f5e-89e1-d8fa5f5db7dc')
dataset

Dataset(id='2ce9cb24-dba0-4f5e-89e1-d8fa5f5db7dc', name='Annual National Land Cover Database', type='Raster', crs=None, description=['This dataset characterises land cover class, land cover change, impervious surface class, fractional impervious surface (%), and spectral change day of year at 30 m spatial resolution. Coverage is for the conterminous United States annually from 1985 onwards, delivered in Albers Equal Area Conic CRS. Pixels are assigned one of 16 land cover classes.'], usage_notes=["New time points are delivered annually. Spatial coverage is limited to the conterminous United States. Land cover values are associated with a July 1st date, so changes after this date appear in the following year's data."], categories=['Land use & land cover'], licence=Licence(type='Open'), version=Version(date='2025-06-25', number='1.1'), spatial_coverage=SpatialCoverage(nominal='Conterminous US'), spatial_resolution=SpatialResolution(nominal='30 m', units='meters', x=30.0, y=30.0), tempora

### Variables

`dataset.variables` is a list of `Variable` objects.

In [4]:
for v in dataset.variables:
    print(v.name)

fractional_impervious_surface
impervious_descriptor
land_cover
land_cover_change
land_cover_confidence
spectral_change_day_of_year


Use `next()` to retrieve a specific variable by name and view the variable-level metadata.

In [5]:
land_cover = next(v for v in dataset.variables if v.name == 'land_cover')
land_cover

Variable(name='land_cover', type='uint8', no_data='250', units='', description=['Index of the predicted land cover class. Predictions are made using deep learning models trained on National Land Cover Database 2019 labels, Landsat data, and a range of ancillary datasets.'], usage_notes=["Indexes represent the surface state on July 1st of each year, so changes after this date appear in the following year's data. Overall accuracy is 82.5%, consistent between years but lower in the eastern US. Highest accuracy is for water (96%, 93% user/producer accuracy) and tree cover (90%, 83%). Lowest accuracy is for wetland (69%, 74%) and barren land (43%, 57%). Out-of-place classifications (particularly developed and barren) may occur over water bodies. Linear artefacts may appear in the desert Southwest due to shrub/scrub and grassland/herbaceous confusion. Not all land cover changes are detectable."], reference_table=[{'Description': 'Open water with < 25% vegetation or soil cover.', 'Index': 11,

### Subscription constraints

`dataset.constraints` describes AOI size limits and geometry requirements. Check these before creating a subscription to ensure your AOI is valid for this dataset.

In [6]:
dataset.constraints

Constraints(aoi_min_hectares=None, aoi_max_hectares=10000000.0, aoi_min_latitude=None, aoi_max_latitude=None, aoi_max_vertices=None, aoi_geometry_types=['MultiPolygon', 'Polygon'], organisation_verified_only=False)

### Reference table

Categorical variables include a `reference_table` — a list of dicts mapping pixel values to class names and descriptions.

In [7]:
reference_table = next(v for v in dataset.variables if v.name == 'land_cover').reference_table
reference_table

[{'Description': 'Open water with < 25% vegetation or soil cover.',
  'Index': 11,
  'Name': 'Open Water'},
 {'Description': 'Permanent ice or snow cover (> 25% total cover).',
  'Index': 12,
  'Name': 'Perennial Ice/Snow'},
 {'Description': 'Mixed constructed materials and vegetation (e.g. lawns, parks, golf courses). Impervious surfaces are < 20% total cover.',
  'Index': 21,
  'Name': 'Developed, Open Space'},
 {'Description': 'Mixed constructed materials and vegetation (e.g. single-family housing). Impervious surfaces are 20%-49% total cover.',
  'Index': 22,
  'Name': 'Developed, Low Intensity'},
 {'Description': 'Mixed constructed materials and vegetation (e.g. single-family housing). Impervious surfaces are 50%-79% total cover.',
  'Index': 23,
  'Name': 'Developed, Medium Intensity'},
 {'Description': 'Highly developed areas (e.g. apartments, commercial, industrial). Impervious surfaces are 80%-100% total cover.',
  'Index': 24,
  'Name': 'Developed, High Intensity'},
 {'Descri

From the reference table object, it's possible to build a lookup to decode pixel values in loaded data, or create legends for visualisations.

In [8]:
class_names = {row['Index']: row['Name'] for row in land_cover.reference_table}
class_names

{11: 'Open Water',
 12: 'Perennial Ice/Snow',
 21: 'Developed, Open Space',
 22: 'Developed, Low Intensity',
 23: 'Developed, Medium Intensity',
 24: 'Developed, High Intensity',
 31: 'Barren Land (Rock/Sand/Clay)',
 41: 'Deciduous Forest',
 42: 'Evergreen Forest',
 43: 'Mixed Forest',
 52: 'Shrub/Scrub',
 71: 'Grassland/ Herbaceous',
 81: 'Pasture/Hay',
 82: 'Cultivated Crops',
 90: 'Woody Wetlands',
 95: 'Emergent Herbaceous Wetlands'}